# Constrained LLM Explanation Generation and Verification

This notebook adds the language-generation layer to the neuro-symbolic clinical-trial matching pipeline.

Goals:

1. Load integrated symbolic patient-trial results.
2. Build a constrained explanation prompt.
3. Generate a natural-language explanation from the symbolic proof.
4. Verify that generated text does not contradict the symbolic result.
5. Preserve MET / NOT_MET / UNKNOWN exactly.

In [1]:
import json
from pathlib import Path

import pandas as pd

In [2]:
PROJECT_ROOT = Path("..")

INTEGRATED_DIR = (
    PROJECT_ROOT
    / "data"
    / "integrated"
)

RULES_DIR = (
    PROJECT_ROOT
    / "data"
    / "rules"
)

SRC_DIR = (
    PROJECT_ROOT
    / "src"
)

print("Integrated:", INTEGRATED_DIR.resolve())
print("Rules:", RULES_DIR.resolve())
print("Source:", SRC_DIR.resolve())

Integrated: D:\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant-Github\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant\data\integrated
Rules: D:\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant-Github\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant\data\rules
Source: D:\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant-Github\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant\src


In [3]:
required_files = [
    INTEGRATED_DIR
    / "integrated_patient_trial_results.json",

    INTEGRATED_DIR
    / "integrated_patient_trial_summary.csv",

    RULES_DIR
    / "patient_trial_rule_results.json"
]

for path in required_files:
    print(
        path.name,
        "→",
        path.exists()
    )

integrated_patient_trial_results.json → True
integrated_patient_trial_summary.csv → True
patient_trial_rule_results.json → True


In [4]:
with open(
    INTEGRATED_DIR
    / "integrated_patient_trial_results.json",
    "r",
    encoding="utf-8"
) as file:
    integrated_results = json.load(file)

print(
    "Patients loaded:",
    len(integrated_results)
)

Patients loaded: 8


In [5]:
integrated_lookup = {}

for patient_entry in integrated_results:

    patient_id = patient_entry[
        "patient_id"
    ]

    for result in patient_entry[
        "results"
    ]:

        integrated_lookup[
            (
                patient_id,
                result["nct_id"]
            )
        ] = result

print(
    "Integrated patient-trial entries:",
    len(integrated_lookup)
)

Integrated patient-trial entries: 40


In [6]:
test_case = integrated_lookup[
    (
        "P004",
        "NCT07144254"
    )
]

print(
    "Status:",
    test_case[
        "overall_status"
    ]
)

print(
    "Symbolic available:",
    test_case[
        "symbolic_available"
    ]
)

print(
    "Questions:",
    len(
        test_case[
            "missing_questions"
        ]
    )
)

Status: UNKNOWN
Symbolic available: True
Questions: 4


In [7]:
def format_rule_for_prompt(rule):

    return (
        f"Rule ID: {rule['rule_id']}\n"
        f"Criterion: {rule['description']}\n"
        f"Patient value: {rule['patient_value']}\n"
        f"Required condition: "
        f"{rule['operator']} "
        f"{rule['expected_value']}"
    )

In [8]:
def build_llm_prompt(
    patient_id,
    nct_id,
    result
):

    if not result["symbolic_available"]:
        raise ValueError(
            "Cannot build symbolic explanation prompt "
            "for a trial without encoded rules."
        )

    explanation_plan = result[
        "explanation_plan"
    ]

    lines = []

    lines.append(
        "You are generating an explanation for a "
        "clinical-trial screening research prototype."
    )

    lines.append("")
    lines.append("STRICT INSTRUCTIONS:")

    lines.append(
        "1. Use only the information provided below."
    )

    lines.append(
        "2. Do not determine eligibility independently."
    )

    lines.append(
        "3. Preserve the symbolic overall status exactly."
    )

    lines.append(
        "4. Do not change MET, NOT_MET, or UNKNOWN."
    )

    lines.append(
        "5. Do not invent patient facts, trial criteria, "
        "or medical evidence."
    )

    lines.append(
        "6. If information is missing, explicitly say "
        "that it is missing."
    )

    lines.append(
        "7. Do not claim final clinical eligibility."
    )

    lines.append(
        "8. State that this result applies only to the "
        "encoded criteria in this prototype."
    )

    lines.append("")
    lines.append(
        f"Patient ID: {patient_id}"
    )

    lines.append(
        f"Trial ID: {nct_id}"
    )

    lines.append(
        "Symbolic overall status: "
        f"{result['overall_status']}"
    )

    lines.append("")
    lines.append(
        "SYMBOLIC CONCLUSION:"
    )

    lines.append(
        explanation_plan["conclusion"]
    )

    lines.append("")
    lines.append("MET CRITERIA:")

    if explanation_plan["supporting_rules"]:
        for rule in explanation_plan[
            "supporting_rules"
        ]:
            lines.append(
                format_rule_for_prompt(rule)
            )
            lines.append("")
    else:
        lines.append("None")

    lines.append("")
    lines.append("NOT_MET CRITERIA:")

    if explanation_plan["failed_rules"]:
        for rule in explanation_plan[
            "failed_rules"
        ]:
            lines.append(
                format_rule_for_prompt(rule)
            )
            lines.append("")
    else:
        lines.append("None")

    lines.append("")
    lines.append("UNKNOWN CRITERIA:")

    if explanation_plan["unknown_rules"]:
        for rule in explanation_plan[
            "unknown_rules"
        ]:
            lines.append(
                format_rule_for_prompt(rule)
            )
            lines.append("")
    else:
        lines.append("None")

    lines.append("")
    lines.append(
        "MISSING-INFORMATION QUESTIONS:"
    )

    if result["missing_questions"]:
        for item in result[
            "missing_questions"
        ]:
            lines.append(
                f"- {item['question']}"
            )
    else:
        lines.append("None")

    lines.append("")
    lines.append("OUTPUT REQUIREMENTS:")

    lines.append(
        "- Write a concise trial-level explanation."
    )

    lines.append(
        "- Mention the overall symbolic status."
    )

    lines.append(
        "- Explain the decisive criteria."
    )

    lines.append(
        "- Mention missing information when applicable."
    )

    lines.append(
        "- Do not introduce any unsupported claims."
    )

    lines.append(
        "- End with: "
        "\"This is not a final clinical eligibility determination.\""
    )

    return "\n".join(lines)

In [9]:
p004_prompt = build_llm_prompt(
    "P004",
    "NCT07144254",
    test_case
)

print(p004_prompt)

You are generating an explanation for a clinical-trial screening research prototype.

STRICT INSTRUCTIONS:
1. Use only the information provided below.
2. Do not determine eligibility independently.
3. Preserve the symbolic overall status exactly.
4. Do not change MET, NOT_MET, or UNKNOWN.
5. Do not invent patient facts, trial criteria, or medical evidence.
6. If information is missing, explicitly say that it is missing.
7. Do not claim final clinical eligibility.
8. State that this result applies only to the encoded criteria in this prototype.

Patient ID: P004
Trial ID: NCT07144254
Symbolic overall status: UNKNOWN

SYMBOLIC CONCLUSION:
The available information is insufficient to determine whether all evaluated criteria are satisfied.

MET CRITERIA:
Rule ID: NCT07144254_R01
Criterion: Osteosarcoma must have histologic verification.
Patient value: True
Required condition: == True

Rule ID: NCT07144254_R02
Criterion: Participant must have relapsed or refractory osteosarcoma.
Patient val

In [10]:
required_prompt_phrases = [
    "Preserve the symbolic overall status exactly",
    "Do not determine eligibility independently",
    "Do not invent patient facts",
    "Symbolic overall status: UNKNOWN",
    "This is not a final clinical eligibility determination"
]

for phrase in required_prompt_phrases:

    print(
        phrase,
        "→",
        phrase in p004_prompt
    )

Preserve the symbolic overall status exactly → True
Do not determine eligibility independently → True
Do not invent patient facts → True
Symbolic overall status: UNKNOWN → True
This is not a final clinical eligibility determination → True


In [11]:
p002_case = integrated_lookup[
    (
        "P002",
        "NCT07560826"
    )
]

p002_prompt = build_llm_prompt(
    "P002",
    "NCT07560826",
    p002_case
)

print(
    p002_prompt[:5000]
)

You are generating an explanation for a clinical-trial screening research prototype.

STRICT INSTRUCTIONS:
1. Use only the information provided below.
2. Do not determine eligibility independently.
3. Preserve the symbolic overall status exactly.
4. Do not change MET, NOT_MET, or UNKNOWN.
5. Do not invent patient facts, trial criteria, or medical evidence.
6. If information is missing, explicitly say that it is missing.
7. Do not claim final clinical eligibility.
8. State that this result applies only to the encoded criteria in this prototype.

Patient ID: P002
Trial ID: NCT07560826
Symbolic overall status: NOT_MET

SYMBOLIC CONCLUSION:
The patient does not satisfy all evaluated eligibility criteria.

MET CRITERIA:
Rule ID: NCT07560826_R01
Criterion: Participant must be 22 years of age or younger.
Patient value: 16
Required condition: <= 22

Rule ID: NCT07560826_R02
Criterion: Participant must have osteosarcoma.
Patient value: osteosarcoma
Required condition: == osteosarcoma

Rule ID: 

In [12]:
p001_case = integrated_lookup[
    (
        "P001",
        "NCT07560826"
    )
]

p001_prompt = build_llm_prompt(
    "P001",
    "NCT07560826",
    p001_case
)

print(
    p001_prompt[:4000]
)

You are generating an explanation for a clinical-trial screening research prototype.

STRICT INSTRUCTIONS:
1. Use only the information provided below.
2. Do not determine eligibility independently.
3. Preserve the symbolic overall status exactly.
4. Do not change MET, NOT_MET, or UNKNOWN.
5. Do not invent patient facts, trial criteria, or medical evidence.
6. If information is missing, explicitly say that it is missing.
7. Do not claim final clinical eligibility.
8. State that this result applies only to the encoded criteria in this prototype.

Patient ID: P001
Trial ID: NCT07560826
Symbolic overall status: MET

SYMBOLIC CONCLUSION:
The patient satisfies all evaluated encoded criteria. This does not establish final clinical eligibility.

MET CRITERIA:
Rule ID: NCT07560826_R01
Criterion: Participant must be 22 years of age or younger.
Patient value: 15
Required condition: <= 22

Rule ID: NCT07560826_R02
Criterion: Participant must have osteosarcoma.
Patient value: osteosarcoma
Required 

In [13]:
def describe_rule(rule):

    return (
        f"{rule['description']} "
        f"(patient value: {rule['patient_value']}; "
        f"required: {rule['operator']} "
        f"{rule['expected_value']})"
    )

In [14]:
def generate_deterministic_explanation(
    patient_id,
    nct_id,
    result
):

    if not result["symbolic_available"]:
        return (
            f"Trial {nct_id} was retrieved as a potentially "
            f"relevant candidate for patient {patient_id}, "
            f"but detailed symbolic eligibility rules have "
            f"not been encoded for this trial. "
            f"No eligibility conclusion is produced. "
            f"This is not a final clinical eligibility determination."
        )

    plan = result["explanation_plan"]

    status = result["overall_status"]

    lines = []

    lines.append(
        f"For patient {patient_id} and trial {nct_id}, "
        f"the symbolic screening status is {status}."
    )

    lines.append(
        plan["conclusion"]
    )

    if status == "MET":

        lines.append(
            f"All {result['met_count']} evaluated encoded "
            f"criteria were satisfied."
        )

    elif status == "NOT_MET":

        lines.append(
            f"{result['not_met_count']} evaluated criterion"
            f"{'' if result['not_met_count'] == 1 else 'ia'} "
            f"were not satisfied."
        )

        if plan["failed_rules"]:

            lines.append(
                "The decisive failed criteria are:"
            )

            for rule in plan["failed_rules"]:
                lines.append(
                    f"- {describe_rule(rule)}"
                )

    elif status == "UNKNOWN":

        lines.append(
            f"{result['unknown_count']} evaluated criterion"
            f"{'' if result['unknown_count'] == 1 else 'ia'} "
            f"could not be resolved from the available "
            f"patient information."
        )

        if plan["unknown_rules"]:

            lines.append(
                "The unresolved criteria are:"
            )

            for rule in plan["unknown_rules"]:
                lines.append(
                    f"- {describe_rule(rule)}"
                )

        if result["missing_questions"]:

            lines.append(
                "Additional information is needed:"
            )

            for item in result["missing_questions"]:
                lines.append(
                    f"- {item['question']}"
                )

    lines.append(
        "This result applies only to the encoded criteria "
        "in this research prototype."
    )

    lines.append(
        "This is not a final clinical eligibility determination."
    )

    return "\n".join(lines)

In [15]:
p001_deterministic = generate_deterministic_explanation(
    "P001",
    "NCT07560826",
    p001_case
)

print(p001_deterministic)

For patient P001 and trial NCT07560826, the symbolic screening status is MET.
The patient satisfies all evaluated encoded criteria. This does not establish final clinical eligibility.
All 18 evaluated encoded criteria were satisfied.
This result applies only to the encoded criteria in this research prototype.
This is not a final clinical eligibility determination.


In [16]:
p002_deterministic = generate_deterministic_explanation(
    "P002",
    "NCT07560826",
    p002_case
)

print(p002_deterministic)

For patient P002 and trial NCT07560826, the symbolic screening status is NOT_MET.
The patient does not satisfy all evaluated eligibility criteria.
4 evaluated criterionia were not satisfied.
The decisive failed criteria are:
- Absolute neutrophil count must be at least 1000/uL. (patient value: 620; required: >= 1000)
- Platelet count must be at least 100000/uL. (patient value: 64000; required: >= 100000)
- Hemoglobin must be at least 8.0 g/dL. (patient value: 7.4; required: >= 8.0)
- Left ventricular ejection fraction must be at least 50 percent. (patient value: 43; required: >= 50)
This result applies only to the encoded criteria in this research prototype.
This is not a final clinical eligibility determination.


In [17]:
p004_deterministic = generate_deterministic_explanation(
    "P004",
    "NCT07144254",
    test_case
)

print(p004_deterministic)

For patient P004 and trial NCT07144254, the symbolic screening status is UNKNOWN.
The available information is insufficient to determine whether all evaluated criteria are satisfied.
4 evaluated criterionia could not be resolved from the available patient information.
The unresolved criteria are:
- At least 14 days must have passed since myelosuppressive chemotherapy. (patient value: None; required: >= 14)
- Creatinine clearance or GFR must be at least 70 mL/min/1.73m2. (patient value: None; required: >= 70)
- ALT must be no more than 5 times ULN. (patient value: None; required: <= 5)
- QTc must be 470 ms or less. (patient value: None; required: <= 470)
Additional information is needed:
- How many days have passed since the patient's last chemotherapy?
- What is the patient's most recent GFR value in mL/min/1.73 m²?
- What is the patient's latest ALT value as a multiple of the upper limit of normal?
- What is the patient's most recent QTc measurement in milliseconds?
This result applie

In [18]:
deterministic_checks = [
    (
        "P001",
        p001_deterministic,
        "MET"
    ),
    (
        "P002",
        p002_deterministic,
        "NOT_MET"
    ),
    (
        "P004",
        p004_deterministic,
        "UNKNOWN"
    )
]

for patient_id, explanation, expected_status in deterministic_checks:

    print(
        patient_id,
        "status preserved:",
        expected_status in explanation,
        "| disclaimer:",
        "This is not a final clinical eligibility determination."
        in explanation
    )

P001 status preserved: True | disclaimer: True
P002 status preserved: True | disclaimer: True
P004 status preserved: True | disclaimer: True


In [19]:
import os

from openai import OpenAI

print("OpenAI package imported successfully")

OpenAI package imported successfully


In [20]:
import os

print("os imported")

os imported


In [21]:
api_key_available = (
    os.environ.get("OPENAI_API_KEY")
    is not None
)

print(
    "API key available:",
    api_key_available
)

API key available: True


In [22]:
client = OpenAI()

print("OpenAI client created")

OpenAI client created


In [23]:
test_response = client.responses.create(
    model="gpt-6-luna",
    input=(
        "Reply with exactly this text: "
        "LLM connection successful"
    )
)

print(
    test_response.output_text
)

LLM connection successful


In [24]:
import re
def extract_declared_status(text):

    clean_text = (
        text
        .replace("**", "")
        .replace("__", "")
    )

    patterns = [
        r"overall symbolic status\s*[:\-]?\s*(NOT_MET|UNKNOWN|MET)",
        r"overall status\s*[:\-]?\s*(NOT_MET|UNKNOWN|MET)",
        r"symbolic screening status\s*(?:is|:)\s*(NOT_MET|UNKNOWN|MET)",
        r"symbolic status\s*(?:is|:)\s*(NOT_MET|UNKNOWN|MET)",
        r"screening status\s*(?:is|:)\s*(NOT_MET|UNKNOWN|MET)"
    ]

    for pattern in patterns:

        match = re.search(
            pattern,
            clean_text,
            flags=re.IGNORECASE
        )

        if match:
            return match.group(1).upper()

    return None

In [25]:
def normalize_text_for_verification(text):
    return (
        str(text)
        .lower()
        .replace(",", "")
        .replace("**", "")
        .replace("__", "")
    )


def verify_explanation(
    explanation,
    expected_status,
    result
):

    checks = {}

    explanation_lower = (
        normalize_text_for_verification(
            explanation
        )
    )

    declared_status = extract_declared_status(
        explanation
    )

    checks["status_correct"] = (
        declared_status == expected_status
    )

    checks["disclaimer_present"] = (
        "this is not a final clinical eligibility determination."
        in explanation_lower
    )

    checks["failed_criteria_supported"] = True
    checks["missing_information_acknowledged"] = True
    checks["unknown_criteria_supported"] = True

    if expected_status == "NOT_MET":

        failed_rules = result[
            "explanation_plan"
        ]["failed_rules"]

        failed_rule_checks = []

        for rule in failed_rules:

            description = (
                normalize_text_for_verification(
                    rule["description"]
                )
            )

            patient_value = (
                normalize_text_for_verification(
                    rule["patient_value"]
                )
            )

            expected_value = (
                normalize_text_for_verification(
                    rule["expected_value"]
                )
            )

            important_terms = [
                word.strip(
                    ".,;:()[]/"
                )
                for word in description.split()
                if len(
                    word.strip(
                        ".,;:()[]/"
                    )
                ) >= 3
            ]

            keyword_match = any(
                term in explanation_lower
                for term in important_terms
            )

            value_match = (
                patient_value
                in explanation_lower
            )

            threshold_match = (
                expected_value
                in explanation_lower
            )

            failed_rule_checks.append(
                keyword_match
                and (
                    value_match
                    or threshold_match
                )
            )

        checks[
            "failed_criteria_supported"
        ] = all(
            failed_rule_checks
        )

    elif expected_status == "UNKNOWN":

        checks[
            "missing_information_acknowledged"
        ] = any(
            phrase in explanation_lower
            for phrase in [
                "missing",
                "additional information",
                "insufficient",
                "unresolved",
                "not available",
                "not provided",
                "cannot be determined"
            ]
        )

        missing_questions = result.get(
            "missing_questions",
            []
        )

        missing_checks = []

        for item in missing_questions:

            missing_field = (
                normalize_text_for_verification(
                    item.get(
                        "missing_field",
                        ""
                    )
                )
            )

            question = (
                normalize_text_for_verification(
                    item.get(
                        "question",
                        ""
                    )
                )
            )

            aliases = []

            combined = (
                missing_field
                + " "
                + question
            )

            if "chemotherapy" in combined:
                aliases.extend([
                    "chemotherapy",
                    "myelosuppressive"
                ])

            if (
                "gfr" in combined
                or
                "creatinine" in combined
            ):
                aliases.extend([
                    "gfr",
                    "creatinine clearance"
                ])

            if "alt" in combined:
                aliases.append("alt")

            if "qtc" in combined:
                aliases.append("qtc")

            aliases = list(
                dict.fromkeys(aliases)
            )

            if aliases:

                missing_checks.append(
                    any(
                        alias
                        in explanation_lower
                        for alias in aliases
                    )
                )

        checks[
            "unknown_criteria_supported"
        ] = (
            all(missing_checks)
            if missing_checks
            else True
        )

    passed = all(
        checks.values()
    )

    return {
        "passed": passed,
        "checks": checks,
        "declared_status": declared_status
    }

In [26]:
def print_verification_result(
    verification
):

    print(
        "Passed:",
        verification["passed"]
    )

    print(
        "Declared status:",
        verification[
            "declared_status"
        ]
    )

    for name, value in verification[
        "checks"
    ].items():

        print(
            name,
            "→",
            value
        )

In [27]:
def generate_llm_explanation(
    prompt,
    model="gpt-6-luna"
):

    response = client.responses.create(
        model=model,
        input=prompt
    )

    return response.output_text.strip()

In [28]:
def generate_verified_explanation(
    patient_id,
    nct_id,
    result,
    prompt,
    model="gpt-6-luna"
):

    deterministic_explanation = (
        generate_deterministic_explanation(
            patient_id,
            nct_id,
            result
        )
    )

    try:

        llm_explanation = (
            generate_llm_explanation(
                prompt,
                model=model
            )
        )

        verification = (
            verify_explanation(
                llm_explanation,
                result["overall_status"],
                result
            )
        )

        if verification["passed"]:

            return {
                "source": "LLM",
                "explanation": llm_explanation,
                "verification": verification,
                "fallback_used": False,
                "error": None
            }

        return {
            "source": "Deterministic Fallback",
            "explanation": deterministic_explanation,
            "verification": verification,
            "fallback_used": True,
            "error": None
        }

    except Exception as error:

        return {
            "source": "Deterministic Fallback",
            "explanation": deterministic_explanation,
            "verification": None,
            "fallback_used": True,
            "error": str(error)
        }

In [29]:
p001_prompt = build_llm_prompt(
    "P001",
    "NCT07560826",
    p001_case
)

p002_prompt = build_llm_prompt(
    "P002",
    "NCT07560826",
    p002_case
)

p004_prompt = build_llm_prompt(
    "P004",
    "NCT07144254",
    test_case
)

print("Prompts rebuilt successfully")

Prompts rebuilt successfully


In [30]:
p001_llm_explanation = (
    generate_llm_explanation(
        p001_prompt
    )
)

print(p001_llm_explanation)

For patient P001 and trial NCT07560826, the overall symbolic status is **MET**. The encoded criteria are satisfied, including age 15 (≤22), high-grade osteosarcoma, localized disease, and the specified MAP treatment. The recorded laboratory and cardiac values meet the encoded thresholds, and the recorded enrollment and exclusion criteria are met. No criteria are listed as NOT_MET or UNKNOWN, and no information is listed as missing.

This result applies only to the encoded criteria in this prototype. This is not a final clinical eligibility determination.


In [31]:
p001_llm_verification = (
    verify_explanation(
        p001_llm_explanation,
        "MET",
        p001_case
    )
)

print_verification_result(
    p001_llm_verification
)

Passed: True
Declared status: MET
status_correct → True
disclaimer_present → True
failed_criteria_supported → True
missing_information_acknowledged → True
unknown_criteria_supported → True


In [32]:
p002_llm_explanation = generate_llm_explanation(
    p002_prompt
)

print(p002_llm_explanation)

For patient P002 and trial NCT07560826, the overall symbolic status is **NOT_MET**: the patient does not satisfy all evaluated eligibility criteria.

The decisive criteria marked NOT_MET are:
- Absolute neutrophil count: 620/uL (required: at least 1000/uL)
- Platelet count: 64,000/uL (required: at least 100,000/uL)
- Hemoglobin: 7.4 g/dL (required: at least 8.0 g/dL)
- Left ventricular ejection fraction: 43% (required: at least 50%)

No missing information was reported. This result applies only to the encoded criteria in this prototype.

This is not a final clinical eligibility determination.


In [33]:
p002_llm_verification = verify_explanation(
    p002_llm_explanation,
    "NOT_MET",
    p002_case
)

print_verification_result(
    p002_llm_verification
)

Passed: True
Declared status: NOT_MET
status_correct → True
disclaimer_present → True
failed_criteria_supported → True
missing_information_acknowledged → True
unknown_criteria_supported → True


In [34]:
p004_llm_explanation = generate_llm_explanation(
    p004_prompt
)

print(p004_llm_explanation)

For patient P004 and trial NCT07144254, the overall symbolic status is **UNKNOWN**. The encoded criteria marked as MET include histologic verification, refractory disease, at least three specified front-line therapies, and the other listed disease, performance, laboratory, respiratory, and treatment-history criteria. No criteria are marked NOT_MET.

The result remains UNKNOWN because information is missing for time since myelosuppressive chemotherapy, creatinine clearance/GFR, ALT, and QTc. This result applies only to the encoded criteria in this prototype.

This is not a final clinical eligibility determination.


In [35]:
p004_llm_verification = verify_explanation(
    p004_llm_explanation,
    "UNKNOWN",
    test_case
)

print_verification_result(
    p004_llm_verification
)

Passed: True
Declared status: UNKNOWN
status_correct → True
disclaimer_present → True
failed_criteria_supported → True
missing_information_acknowledged → True
unknown_criteria_supported → True


In [36]:
p001_safe_result = (
    generate_verified_explanation(
        "P001",
        "NCT07560826",
        p001_case,
        p001_prompt
    )
)

print(
    "Source:",
    p001_safe_result["source"]
)

print(
    "Fallback used:",
    p001_safe_result[
        "fallback_used"
    ]
)

print()

print(
    p001_safe_result[
        "explanation"
    ]
)

Source: LLM
Fallback used: False

**Overall symbolic status: MET.** All evaluated encoded criteria are marked MET, with no NOT_MET or UNKNOWN criteria and no missing-information questions.

The recorded values meet the encoded age, osteosarcoma type and grade, disease-status, body-surface-area, MAP treatment and dose, laboratory, and cardiac-function conditions. The pregnancy and breastfeeding exclusion rules are also marked MET. The enrollment-timing rule is marked MET with a recorded value of `False` and required condition `== False`.

This result applies only to the encoded criteria in this prototype. This is not a final clinical eligibility determination.


In [37]:
p002_safe_result = (
    generate_verified_explanation(
        "P002",
        "NCT07560826",
        p002_case,
        p002_prompt
    )
)

print(
    "Source:",
    p002_safe_result["source"]
)

print(
    "Fallback used:",
    p002_safe_result[
        "fallback_used"
    ]
)

print()

print(
    p002_safe_result[
        "explanation"
    ]
)

Source: Deterministic Fallback
Fallback used: True

For patient P002 and trial NCT07560826, the symbolic screening status is NOT_MET.
The patient does not satisfy all evaluated eligibility criteria.
4 evaluated criterionia were not satisfied.
The decisive failed criteria are:
- Absolute neutrophil count must be at least 1000/uL. (patient value: 620; required: >= 1000)
- Platelet count must be at least 100000/uL. (patient value: 64000; required: >= 100000)
- Hemoglobin must be at least 8.0 g/dL. (patient value: 7.4; required: >= 8.0)
- Left ventricular ejection fraction must be at least 50 percent. (patient value: 43; required: >= 50)
This result applies only to the encoded criteria in this research prototype.
This is not a final clinical eligibility determination.


In [38]:
p004_safe_result = (
    generate_verified_explanation(
        "P004",
        "NCT07144254",
        test_case,
        p004_prompt
    )
)

print(
    "Source:",
    p004_safe_result["source"]
)

print(
    "Fallback used:",
    p004_safe_result[
        "fallback_used"
    ]
)

print()

print(
    p004_safe_result[
        "explanation"
    ]
)

Source: LLM
Fallback used: False

For patient P004 and trial NCT07144254, the overall symbolic status is **UNKNOWN**. The evaluated criteria marked MET include histologic verification, refractory disease, at least three specified front-line therapies, measurable or evaluable disease, and the recorded performance, blood-count, bilirubin, respiratory, infection, and prior-treatment criteria. No criteria are marked NOT_MET.

The status remains UNKNOWN because information is missing for time since myelosuppressive chemotherapy, creatinine clearance/GFR, ALT, and QTc. These values are needed to evaluate the corresponding encoded criteria.

This result applies only to the encoded criteria in this prototype. This is not a final clinical eligibility determination.


In [39]:
print(
    p002_safe_result["verification"]
)

{'passed': False, 'checks': {'status_correct': False, 'disclaimer_present': True, 'failed_criteria_supported': True, 'missing_information_acknowledged': True, 'unknown_criteria_supported': True}, 'declared_status': None}


In [40]:
print(
    p004_safe_result["verification"]
)

{'passed': True, 'checks': {'status_correct': True, 'disclaimer_present': True, 'failed_criteria_supported': True, 'missing_information_acknowledged': True, 'unknown_criteria_supported': True}, 'declared_status': 'UNKNOWN'}


In [41]:
print("P002")
print_verification_result(
    p002_safe_result["verification"]
)

print()

print("P004")
print_verification_result(
    p004_safe_result["verification"]
)

P002
Passed: False
Declared status: None
status_correct → False
disclaimer_present → True
failed_criteria_supported → True
missing_information_acknowledged → True
unknown_criteria_supported → True

P004
Passed: True
Declared status: UNKNOWN
status_correct → True
disclaimer_present → True
failed_criteria_supported → True
missing_information_acknowledged → True
unknown_criteria_supported → True


In [42]:
EXPLANATIONS_DIR = (
    PROJECT_ROOT
    / "data"
    / "explanations"
)

EXPLANATIONS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "Explanations folder:",
    EXPLANATIONS_DIR.resolve()
)

Explanations folder: D:\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant-Github\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant\data\explanations


In [43]:
verified_showcase_results = [
    {
        "patient_id": "P001",
        "nct_id": "NCT07560826",
        "expected_status": "MET",
        "llm_explanation": p001_llm_explanation,
        "verification_passed": p001_llm_verification["passed"],
        "declared_status": p001_llm_verification["declared_status"],
        "verification_checks": p001_llm_verification["checks"]
    },
    {
        "patient_id": "P002",
        "nct_id": "NCT07560826",
        "expected_status": "NOT_MET",
        "llm_explanation": p002_llm_explanation,
        "verification_passed": p002_llm_verification["passed"],
        "declared_status": p002_llm_verification["declared_status"],
        "verification_checks": p002_llm_verification["checks"]
    },
    {
        "patient_id": "P004",
        "nct_id": "NCT07144254",
        "expected_status": "UNKNOWN",
        "llm_explanation": p004_llm_explanation,
        "verification_passed": p004_llm_verification["passed"],
        "declared_status": p004_llm_verification["declared_status"],
        "verification_checks": p004_llm_verification["checks"]
    }
]

In [44]:
verified_json_path = (
    EXPLANATIONS_DIR
    / "verified_llm_explanations.json"
)

with open(
    verified_json_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        verified_showcase_results,
        file,
        indent=2
    )

print(
    "Saved:",
    verified_json_path.resolve()
)

Saved: D:\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant-Github\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant\data\explanations\verified_llm_explanations.json


In [45]:
verification_summary_df = pd.DataFrame([
    {
        "patient_id": item["patient_id"],
        "nct_id": item["nct_id"],
        "expected_status": item["expected_status"],
        "declared_status": item["declared_status"],
        "verification_passed": item["verification_passed"]
    }
    for item in verified_showcase_results
])

verification_summary_df

,patient_id,nct_id,expected_status,declared_status,verification_passed
0,P001,NCT07560826,MET,MET,True
1,P002,NCT07560826,NOT_MET,NOT_MET,True
2,P004,NCT07144254,UNKNOWN,UNKNOWN,True


In [46]:
verification_csv_path = (
    EXPLANATIONS_DIR
    / "verified_llm_explanations_summary.csv"
)

verification_summary_df.to_csv(
    verification_csv_path,
    index=False
)

print(
    "Saved:",
    verification_csv_path.resolve()
)

Saved: D:\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant-Github\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant\data\explanations\verified_llm_explanations_summary.csv


In [47]:
for path in [
    verified_json_path,
    verification_csv_path
]:
    print(
        path.name,
        "→",
        path.exists()
    )

verified_llm_explanations.json → True
verified_llm_explanations_summary.csv → True


## Notebook 05 Conclusion

This notebook added constrained LLM explanation generation and verification to the neuro-symbolic trial-matching pipeline.

Completed functionality:

- Built structured prompts from symbolic proof traces.
- Generated deterministic fallback explanations.
- Connected an external LLM for natural-language explanation generation.
- Preserved MET / NOT_MET / UNKNOWN outcomes.
- Verified that LLM explanations matched the symbolic result.
- Verified support for failed criteria and missing-information cases.
- Added deterministic fallback behavior for invalid or rejected LLM outputs.
- Validated representative MET, NOT_MET, and UNKNOWN cases.
- Saved verified explanation outputs for reuse in the frontend.

The next stage is to connect the integrated neuro-symbolic pipeline and verified explanation layer to the Streamlit user interface.